In [1]:
%pip install lightgbm xlrd


Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd

# Taiwan credit default dataset
taiwan = pd.read_excel('data/default of credit card clients.xls', header=1)

# German credit dataset
german = pd.read_csv('data/german.data', sep=' ', header=None)

print("Taiwan shape:", taiwan.shape)
print("Taiwan target distribution:")
print(taiwan['default payment next month'].value_counts(normalize=True))

print("\nGerman shape:", german.shape)
print("German target distribution:")
print(german[20].value_counts(normalize=True))

Taiwan shape: (30000, 25)
Taiwan target distribution:
default payment next month
0    0.7788
1    0.2212
Name: proportion, dtype: float64

German shape: (1000, 21)
German target distribution:
20
1    0.7
2    0.3
Name: proportion, dtype: float64


In [3]:
import numpy as np

# --- Taiwan ---
taiwan_clean = taiwan.drop(columns=['ID']).copy()
taiwan_clean = taiwan_clean.rename(columns={'default payment next month': 'default'})

# Fold undocumented codes into "others"
taiwan_clean['EDUCATION'] = taiwan_clean['EDUCATION'].replace({0: 4, 5: 4, 6: 4})
taiwan_clean['MARRIAGE'] = taiwan_clean['MARRIAGE'].replace({0: 3})

print("EDUCATION now:", sorted(taiwan_clean['EDUCATION'].unique()))
print("MARRIAGE now:", sorted(taiwan_clean['MARRIAGE'].unique()))

# --- German ---
german_cols = ['checking_status','duration','credit_history','purpose','credit_amount',
               'savings_status','employment','installment_rate','personal_status_sex',
               'other_parties','residence_since','property_magnitude','age',
               'other_payment_plans','housing','existing_credits','job',
               'num_dependents','telephone','foreign_worker','target']

german_clean = german.copy()
german_clean.columns = german_cols

# Recode target so bad credit = 1 (matches Taiwan, where default = 1)
german_clean['target'] = german_clean['target'].map({1: 0, 2: 1})

print("\nGerman target now:", german_clean['target'].value_counts(normalize=True).to_dict())
print("German shape:", german_clean.shape)

EDUCATION now: [np.int64(1), np.int64(2), np.int64(3), np.int64(4)]
MARRIAGE now: [np.int64(1), np.int64(2), np.int64(3)]

German target now: {0: 0.7, 1: 0.3}
German shape: (1000, 21)


In [4]:
from sklearn.model_selection import train_test_split

# --- German: one-hot encode the categorical columns ---
X_german = pd.get_dummies(german_clean.drop(columns=['target']), drop_first=False)
y_german = german_clean['target']

print("German features after encoding:", X_german.shape)

# --- Taiwan: already numeric ---
X_taiwan = taiwan_clean.drop(columns=['default'])
y_taiwan = taiwan_clean['default']

print("Taiwan features:", X_taiwan.shape)

# --- Stratified 80/20 splits ---
Xg_train, Xg_test, yg_train, yg_test = train_test_split(
    X_german, y_german, test_size=0.2, stratify=y_german, random_state=42)

Xt_train, Xt_test, yt_train, yt_test = train_test_split(
    X_taiwan, y_taiwan, test_size=0.2, stratify=y_taiwan, random_state=42)

print("\nGerman train/test:", Xg_train.shape, Xg_test.shape)
print("German bad rate - train: %.3f, test: %.3f" % (yg_train.mean(), yg_test.mean()))
print("Taiwan train/test:", Xt_train.shape, Xt_test.shape)
print("Taiwan default rate - train: %.3f, test: %.3f" % (yt_train.mean(), yt_test.mean()))

German features after encoding: (1000, 61)
Taiwan features: (30000, 23)

German train/test: (800, 61) (200, 61)
German bad rate - train: 0.300, test: 0.300
Taiwan train/test: (24000, 23) (6000, 23)
Taiwan default rate - train: 0.221, test: 0.221


In [5]:
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier

# --- Random Forest ---
rf_german = RandomForestClassifier(n_estimators=300, random_state=42, class_weight='balanced')
rf_german.fit(Xg_train, yg_train)

rf_taiwan = RandomForestClassifier(n_estimators=300, random_state=42, class_weight='balanced')
rf_taiwan.fit(Xt_train, yt_train)

# --- LightGBM ---
lgb_german = LGBMClassifier(n_estimators=300, random_state=42, verbose=-1)
lgb_german.fit(Xg_train, yg_train)

lgb_taiwan = LGBMClassifier(n_estimators=300, random_state=42, verbose=-1)
lgb_taiwan.fit(Xt_train, yt_train)

print("All four models trained.")

# Quick sanity check on predicted probabilities
for name, model, X in [("RF German", rf_german, Xg_test),
                       ("LGBM German", lgb_german, Xg_test),
                       ("RF Taiwan", rf_taiwan, Xt_test),
                       ("LGBM Taiwan", lgb_taiwan, Xt_test)]:
    probs = model.predict_proba(X)[:, 1]
    print(f"{name}: mean predicted risk = {probs.mean():.3f}")

All four models trained.
RF German: mean predicted risk = 0.300
LGBM German: mean predicted risk = 0.267
RF Taiwan: mean predicted risk = 0.221
LGBM Taiwan: mean predicted risk = 0.216


In [6]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, average_precision_score, confusion_matrix)

def evaluate(name, dataset, model, X_test, y_test):
    preds = model.predict(X_test)
    probs = model.predict_proba(X_test)[:, 1]
    tn, fp, fn, tp = confusion_matrix(y_test, preds).ravel()
    return {
        'Dataset': dataset, 'Model': name,
        'Accuracy': round(accuracy_score(y_test, preds), 3),
        'Precision': round(precision_score(y_test, preds), 3),
        'Recall': round(recall_score(y_test, preds), 3),
        'F1': round(f1_score(y_test, preds), 3),
        'ROC-AUC': round(roc_auc_score(y_test, probs), 3),
        'PR-AUC': round(average_precision_score(y_test, probs), 3),
        'TP': tp, 'FP': fp, 'FN': fn, 'TN': tn
    }

results = pd.DataFrame([
    evaluate('Random Forest', 'German', rf_german, Xg_test, yg_test),
    evaluate('LightGBM', 'German', lgb_german, Xg_test, yg_test),
    evaluate('Random Forest', 'Taiwan', rf_taiwan, Xt_test, yt_test),
    evaluate('LightGBM', 'Taiwan', lgb_taiwan, Xt_test, yt_test),
])
print(results.to_string(index=False))

# The "no model at all" baseline
print("\nBaseline (predict nobody defaults):")
print("  German accuracy: %.3f, recall: 0.000" % (1 - yg_test.mean()))
print("  Taiwan accuracy: %.3f, recall: 0.000" % (1 - yt_test.mean()))

# German cost matrix: missing a bad customer costs 5x a false alarm
print("\nGerman cost (5x FN + 1x FP):")
for name, model in [('Random Forest', rf_german), ('LightGBM', lgb_german)]:
    tn, fp, fn, tp = confusion_matrix(yg_test, model.predict(Xg_test)).ravel()
    print(f"  {name}: {5*fn + fp}  (FN={fn}, FP={fp})")

Dataset         Model  Accuracy  Precision  Recall    F1  ROC-AUC  PR-AUC  TP  FP  FN   TN
 German Random Forest     0.755      0.667   0.367 0.473    0.795   0.629  22  11  38  129
 German      LightGBM     0.765      0.633   0.517 0.569    0.776   0.621  31  18  29  122
 Taiwan Random Forest     0.812      0.640   0.342 0.446    0.760   0.538 454 255 873 4418
 Taiwan      LightGBM     0.810      0.627   0.347 0.447    0.768   0.532 461 274 866 4399

Baseline (predict nobody defaults):
  German accuracy: 0.700, recall: 0.000
  Taiwan accuracy: 0.779, recall: 0.000

German cost (5x FN + 1x FP):
  Random Forest: 201  (FN=38, FP=11)
  LightGBM: 163  (FN=29, FP=18)


In [7]:
import matplotlib.pyplot as plt

def top_features(model, X, n=10):
    return pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False).head(n)

print("=== GERMAN — Random Forest ===")
print(top_features(rf_german, Xg_train).to_string())

print("\n=== GERMAN — LightGBM ===")
print(top_features(lgb_german, Xg_train).to_string())

print("\n=== TAIWAN — Random Forest ===")
print(top_features(rf_taiwan, Xt_train).to_string())

print("\n=== TAIWAN — LightGBM ===")
print(top_features(lgb_taiwan, Xt_train).to_string())

=== GERMAN — Random Forest ===
credit_amount          0.089959
duration               0.078710
age                    0.073447
checking_status_A14    0.065272
installment_rate       0.035725
checking_status_A11    0.032851
residence_since        0.031594
credit_history_A34     0.025595
savings_status_A61     0.022522
checking_status_A12    0.020001

=== GERMAN — LightGBM ===
credit_amount              2099
age                        1154
duration                    848
installment_rate            369
residence_since             336
checking_status_A11         189
credit_history_A32          179
existing_credits            172
personal_status_sex_A92     160
personal_status_sex_A93     158

=== TAIWAN — Random Forest ===
PAY_0        0.094165
LIMIT_BAL    0.064097
AGE          0.062706
BILL_AMT1    0.061595
BILL_AMT2    0.053986
PAY_AMT1     0.053286
PAY_AMT2     0.051769
BILL_AMT3    0.050037
BILL_AMT4    0.049623
PAY_AMT3     0.049106

=== TAIWAN — LightGBM ===
AGE          708
BILL_A